# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore and process the FAIR$^2$ tabular dataset for second primary colorectal cancer. All dataset schema entities (record sets, fields, columns) are referenced by their unique `@id` fields, as recommended for FAIR-compliant usage.

### Dataset Source
The dataset's Croissant schema is available at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print main metadata as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Display available record sets, their `@id` values, and their fields (with `@id`).

Each Croissant record set typically represents a logical table or entity in the dataset. All entity references use their `@id`.

In [ ]:
# List available RecordSets with their @id
record_sets = list(dataset.record_sets)

print("Available Record Sets and their fields:")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    {field.name}: {field.id} (dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load records from a selected record set into a pandas DataFrame for further exploration.

For this analysis, we'll use the main patient record set (identified by its `@id`). Below, all entity access will use the canonical `@id` string from the previous cell.

In [ ]:
# List available RecordSet @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load all records for each record set into separate DataFrames
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded RecordSet: {rsid} ({len(df)} rows)")

# We'll select the main clinical/patient record set for example exploration:
# For this dataset (2024-06 FAIR2), look for a RecordSet with >60 records and fields about clinical variables. Print columns as reference.
main_record_set_id = None
main_record_set_columns = None
for rsid, df in dataframes.items():
    if df.shape[0] >= 70 and df.shape[1] > 5:
        main_record_set_id = rsid
        main_record_set_columns = df.columns.tolist()
        print(f"Main clinical RecordSet selected: {main_record_set_id}")
        break
if main_record_set_columns:
    print("Columns (fields @id) available:", main_record_set_columns)
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply typical EDA operations:
- Filter records (e.g., by numeric clinical variable)
- Normalize numeric fields
- Group and summarize by a clinical attribute

All fields referenced below use their `@id` from the schema to comply with FAIR best practices.

In [ ]:
# Example: Choose a numeric field (e.g., 'Age' or similar). Use column id from above.
# Replace with the corresponding @id string for age at diagnosis or similar numeric variable, if available.
selected_numeric_field = None
# Try to select a likely numeric clinical field
for col in dataframes[main_record_set_id].columns:
    if 'Age' in col or 'age' in col or 'Interval' in col or 'interval' in col:
        selected_numeric_field = col
        print(f"Numeric field chosen: {selected_numeric_field}")
        break
if not selected_numeric_field:
    # fallback: just use the first column if not found
    selected_numeric_field = dataframes[main_record_set_id].columns[0]

threshold = 50  # e.g., age > 50
try:
    filtered_df = dataframes[main_record_set_id][
        pd.to_numeric(dataframes[main_record_set_id][selected_numeric_field], errors='coerce') > threshold
    ].copy()
except Exception:
    # fallback: no filtering
    filtered_df = dataframes[main_record_set_id]

print(f"Filtered records with {selected_numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{selected_numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[selected_numeric_field], errors='coerce') - pd.to_numeric(filtered_df[selected_numeric_field], errors='coerce').mean()
) / pd.to_numeric(filtered_df[selected_numeric_field], errors='coerce').std()
print(f"Normalized {selected_numeric_field} for filtered records:")
print(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized"]].head())

# Group by an available clinical field, e.g., sex, site, or MSI status (all using @id for columns)
group_field = None
for col in dataframes[main_record_set_id].columns:
    if 'sex' in col.lower() or 'Sex' in col or 'MSI' in col or 'msi' in col or 'site' in col or 'Site' in col:
        group_field = col
        print(f"Grouping by field: {group_field}")
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field, dropna=False)[selected_numeric_field].mean().reset_index().sort_values(selected_numeric_field, ascending=False)
    print(f"Grouped (mean of {selected_numeric_field}) by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

- Histogram/distribution for the selected numeric field
- Grouped barplot for mean value grouped by another field (sex/MSI/site/etc)

In [ ]:
# Visualization: Distribution of the selected numeric field
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
pd.to_numeric(dataframes[main_record_set_id][selected_numeric_field], errors='coerce').hist(bins=15)
plt.title(f"Distribution of {selected_numeric_field}")
plt.xlabel(selected_numeric_field)
plt.ylabel("Count")
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

# If grouped summary exists, plot bar plot
if group_field and 'grouped_df' in locals():
    plt.figure(figsize=(7,4))
    plt.bar(grouped_df[group_field].astype(str), grouped_df[selected_numeric_field])
    plt.title(f"Mean {selected_numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {selected_numeric_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

- This notebook demonstrates programmatic data loading and exploratory analysis for the FAIR$^2$ colorectal cohort using the `mlcroissant` standard and strict usage of `@id` references for all records and fields.
- The dataset enables stratified analysis by clinical characteristics (e.g., age threshold, molecular subtype) for advanced research on second primary colorectal cancer cases, compliant with machine-actionable metadata.

> For more advanced usage, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).